## Importing Data:

In [30]:
import pandas as pd 

path = '/kaggle/input/competitions/playground-series-s6e4/'

train_df = pd.read_csv(path + 'train.csv')
display(train_df.head())
print('Training set: ', train_df.shape)

test_df = pd.read_csv(path + 'test.csv')
display(test_df.head())
print('Test set: ', test_df.shape)

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


Training set:  (630000, 21)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
1,630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
2,630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
3,630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
4,630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


Test set:  (270000, 20)


In [31]:
target = 'Irrigation_Need'

X = train_df.drop(columns=['id', target], axis=1)
y = train_df[target].map({'Low': 0, 'Medium': 1, 'High': 2})

## Feature Engineering:



In [32]:
num_cols = X.select_dtypes(include='number').columns
cat_cols = X.select_dtypes("object").columns

In [33]:
X[cat_cols] = X[cat_cols].astype('category')
test_df[cat_cols] = test_df[cat_cols].astype('category')

### Frequency Encoding:
From EDA we saw mostly all categories in our categorical features had near-equal counts. If all categoricals are balanced like that, frequency encoding adds near-zero signal since all values map to ~same frequency. So we will be skiping frequency encoding for the same reason, just wanted to point out why we wont use this technique here.

### Feature Corsses:
Writting a custom transformer just to keep it clean

In [34]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()

        # Convert to string first
        cat_cols = X.select_dtypes(include=['category', 'object']).columns
        X[cat_cols] = X[cat_cols].astype(str)
        
        # Create _orig copies
        for col in cat_cols:
            X[f'{col}_orig'] = X[col]
        
        # Numerical crosses
        X['heat_humidity_stress'] = X['Temperature_C'] * (1 - X['Humidity']/100)
        X['water_availability']   = X['Rainfall_mm'] + X['Previous_Irrigation_mm']
        X['evaporation_proxy']    = X['Temperature_C'] * X['Wind_Speed_kmh'] * X['Sunlight_Hours']
        X['soil_health']          = X['Organic_Carbon'] * X['Soil_Moisture']
        
        # ← Fix: convert to string first before concatenating
        X['region_season']     = X['Region'].astype(str) + '_' + X['Season'].astype(str)
        X['crop_growth']       = X['Crop_Type'].astype(str) + '_' + X['Crop_Growth_Stage'].astype(str)
        X['soil_crop']         = X['Soil_Type'].astype(str) + '_' + X['Crop_Type'].astype(str)
        X['source_irrigation'] = X['Water_Source'].astype(str) + '_' + X['Irrigation_Type'].astype(str)
        
        # Season cyclical
        season_map = {'Spring':0, 'Summer':1, 'Autumn':2, 'Winter':3}
        X['season_num'] = X['Season'].astype(str).map(season_map)
        X['season_sin'] = np.sin(2 * np.pi * X['season_num'] / 4)
        X['season_cos'] = np.cos(2 * np.pi * X['season_num'] / 4)
        X.drop('season_num', axis=1, inplace=True)
        
        return X

### Target Encoding:
Target Encoding = replacing a categorical value with the mean of the target for that category.

In [35]:
import category_encoders as ce

#something like this will be used in the pipeline
target_encoder = ce.TargetEncoder(
         cols=cat_cols,
        smoothing=10 #prevents overfitting
)

## Baseline Test Model:
This is a baseline XGBoost model to check if our created features are being used or no via feature importance.

In [36]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer

# Cols that TargetEncoder will handle
te_cols = cat_cols.tolist() + ['region_season', 'crop_growth', 
                                'soil_crop', 'source_irrigation']

# Cols that are copies → use OrdinalEncoder for XGBoost
orig_cols = [f'{col}_orig' for col in cat_cols]

preprocessor = ColumnTransformer(transformers=[
    ('te',      ce.TargetEncoder(),   te_cols),
    ('ordinal', OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1), orig_cols),
], remainder='passthrough')  # passes numerical cols through untouched

pipeline = Pipeline(steps=[
    ('feature_engineer', FeatureEngineer()),
    ('preprocessor', preprocessor),
    ('model',XGBClassifier(
                n_estimators=600,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                device='cuda',
                enable_categorical=True
        ))
])

In [37]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_score = cross_val_score(pipeline, X, y, cv=skf, scoring='balanced_accuracy')
print('cross_val_score: ',cv_score)
print('mean_val_score: ', cv_score.mean())
print("std_val_score",cv_score.std())


cross_val_score:  [0.96143235 0.96366466 0.96309264 0.96250021 0.96146693]
mean_val_score:  0.9624313573812564
std_val_score 0.0008821801494398271


In [38]:
pipeline.fit(X,y)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('feature_engineer', FeatureEngineer()),
                ('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('te', TargetEncoder(),
                                                  ['Soil_Type', 'Crop_Type',
                                                   'Crop_Growth_Stage',
                                                   'Season', 'Irrigation_Type',
                                                   'Water_Source',
                                                   'Mulching_Used', 'Region',
                                                   'region_season',
                                                   'crop_growth', 'soil_crop',
                                                   'source_irrigation']),
                                                 ('ordinal',
                                                  OrdinalEncoder...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=6, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=600, n_jobs=None,
                               num_parallel_tree=None, ...))])

## Predicting on Test data:

In [40]:
X_test = test_df.drop('id', axis=1)

y_preds = pipeline.predict(X_test)
final_preds = pd.Series(y_preds).map({0: 'Low', 1: 'Medium', 2: 'High'})

In [41]:
# submission csv
submission = pd.DataFrame({
    'id': test_df['id'],
    target: final_preds
})

submission.to_csv("submission.csv", index=False)